# PROC GLMSELECT를 활용한 수요 동인 변수 탐색: 단계적 선택, LASSO, 검증된 전진선택법

## 요약

소매 분석팀은 **PROC GLMSELECT**를 사용하여 어떤 마케팅 및 가격 레버가 실제로 주간 판매량을 움직이는지 파악하고, 진짜 수요 동인 변수를 잡음과 분리합니다. SBC로 평가한 단계적 선택법, 5-폴드 교차검증을 사용한 LASSO, 홀드아웃으로 검증한 전진선택법 모두 **동일한 참 동인 변수 집합**—자사가격, 경쟁사가격, 광고비, 이메일 발송량, 프로모션, 공휴일, 북동부(Northeast) 지역, 온라인(Online) 채널—을 회복하며, 심어 놓은 네 개의 가짜 변수(`temp_f`, `noise1`, `noise2`, `noise3`)는 모두 자동으로 제외됩니다.

세 방법은 효과 크기에서도 서로 밀접하게 일치합니다. 각 방법은 자사가격 효과를 달러당 **약 -28 단위**로, 경쟁사가격 효과를 **약 +14**로 추정하는데, 이는 데이터 생성 방정식에 내장된 대체 관계의 부호와 정확히 일치합니다. 유일한 불일치는 한계 영역에서 나타납니다—교차검증된 LASSO는 통계적으로 미미한 `Region=Midwest` 대비항(추정값 -8.3, 표준오차 8.3)을 추가로 유지하는 반면, 단계적 선택법과 전진선택법은 이를 모두 제외합니다. 서로 다른 세 가지 선택 철학을 모두 통과한 동인 변수 목록은 하나의 규칙에만 맞춰 조정된 목록보다 훨씬 신뢰할 수 있습니다.

## 데이터 출처

이 노트북의 모든 데이터는 **합성 데이터**로 인라인에서 생성됩니다(외부 파일 없음, 시드 `20260531`). 소비재 소매업체의 매장-주차(store-week) 수요 패널 한 시즌을 모사합니다.

| 데이터셋 | 행 수 | 단위 | 주요 변수 |
|---------|------|-------|---------------|
| `demand` | 100 | 매장-주차 | `units`(반응변수: 주간 판매량); `price`, `competitor`(자사 및 경쟁사 판매가격); `adspend`, `email`(광고 강도); `promo`, `holiday`(이벤트 플래그); `region`, `channel`(CLASS 효과); `temp_f`, `noise1`-`noise3`(가짜/무관 예측변수) |

`units`는 알려진 선형 방정식으로부터 생성되므로 "정답" 동인 변수 집합을 검증할 수 있습니다. `temp_f`와 세 개의 `noise` 열은 실제 신호를 전혀 담고 있지 않으며, 각 선택 방법이 이들을 제대로 제외하는지 검증하기 위해 존재합니다.

# PROC GLMSELECT를 활용한 수요 동인 변수 탐색

카테고리 매니저에게는 주간 판매량을 설명할 수 있는 후보 변수가 수십 개나 있습니다: 자사가격, 경쟁사가격, 광고비, 이메일 발송량, 프로모션, 공휴일, 매장 지역, 판매채널, 심지어 날씨까지도요. 이 모든 것을 하나의 회귀모형에 던져 넣으면 과적합과 불안정한 계수 추정이 발생하기 쉽습니다. **PROC GLMSELECT**는 내장된 교차검증과 홀드아웃 분할 기능을 갖추고, 단계적 선택법, LASSO, 엘라스틱넷, 최소각회귀 선택법을 지원하여 간결한 모형을 찾는 과정을 자동화합니다.

이 노트북에서는 다음을 수행합니다.

1. *알려진* 참 동인 변수 집합(및 의도적인 가짜 변수)을 가진 현실적인 합성 매장-주차 수요 패널을 생성합니다.
2. SBC로 평가하는 **단계적 선택법**을 실행합니다.
3. 5-폴드 교차검증을 사용하는 **LASSO**를 실행합니다.
4. **30% 홀드아웃으로 검증하는 전진선택법**을 실행합니다.

좋은 선택 방법이라면 실제 동인 변수를 회복하고 잡음을 제외해야 합니다—확인해 보겠습니다.

## 1. 합성 수요 패널 생성

반응변수 `units`는 명시적인 선형 방정식으로 구성되므로 실제 정답을 알고 있습니다: 자사가격과 경쟁사가격, 광고비, 이메일 발송량, 프로모션 및 공휴일 플래그, 그리고 지역 및 채널 효과가 모두 영향을 미칩니다. 변수 `temp_f`, `noise1`, `noise2`, `noise3`는 판매량과 실제 관계가 전혀 없는 순수한 가짜 변수입니다. `call streaminit` 시드를 사용해 데이터를 재현 가능하게 만듭니다.

In [1]:
/* ---------------------------------------------------------------
   소비재 소매업체를 위한 합성 매장-주차 수요 패널.
   'units'는 알려진 방정식을 따르며; temp_f와 noise1-3는 가짜 변수입니다.
   --------------------------------------------------------------- */
데이터 demand;
    호출 streaminit(20260531);
    길이 region $9 channel $8 promo $3;
    반복 store_week = 1 까지 100;
        /* 지역 구성 */
        u1 = rand('uniform');
        만약 u1 < 0.34 이면 region = 'Northeast';
        아니면 만약 u1 < 0.67 이면 region = 'Midwest';
        아니면 region = 'South';

        /* 판매채널 */
        만약 rand('uniform') < 0.45 이면 channel = 'Store';
        아니면 channel = 'Online';

        /* 가격 및 미디어 동인 변수 */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* 이벤트 플래그 및 무관한 날씨 가짜 변수 */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        만약 rand('uniform') < 0.40 이면 promo = 'Yes';
        아니면 promo = 'No';

        /* 순수 잡음 예측변수(참 효과 없음) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* 알려진 구조 방정식으로부터 산출된 주간 판매량 */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        만약 units < 0 이면 units = 0;
        출력;
    종료;
    유지 region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
실행;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. 데이터 프로파일링

모델링에 앞서, 반응변수와 주요 연속형 후보 변수들이 합리적인 척도에 있는지 확인합니다.

In [2]:
처리 평균 데이터=demand n mean std MIN MAX maxdec=1;
    변수 units price competitor adspend email;
    라벨 units="주간 판매량" price="자사가격" competitor="경쟁사가격"
          adspend="광고비" email="이메일 발송량";
    제목 "주간 수요 및 후보 동인 변수";
실행;

                                                    주간 수요 및 후보 동인 변수                                                    

                                                  The MEANS Procedure

 Variable    Label                       N        Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------
 units       주간 판매량                    100       874.8       136.3       508.6      1162.3
 price       자사가격                      100        14.0         3.4         8.0        19.9
 competitor  경쟁사가격                     100        13.8         3.4         8.1        19.9
 adspend     광고비                       100       990.6       726.9        50.0      3358.0
 email       이메일 발송량                   100        45.5        26.4         0.0        99.0
 -----------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. SBC로 평가하는 단계적 선택법

단계적 선택법은 효과를 번갈아 추가하고 제거하는데, 여기서는 진입 검정(`select=sbc`)과 최종 모형 선택(`choose=sbc`) 모두에 **슈바르츠 베이지안 기준(SBC)**을 사용합니다. SBC는 AIC보다 복잡도에 더 큰 벌점을 부과하여 더 간결한 모형을 선호합니다.

주요 문(statement)과 옵션:

- `CLASS region channel promo / param=reference`는 범주형 예측변수를 참조셀 코딩으로 선언합니다.
- `selection=stepwise(select=sbc choose=sbc)`가 탐색을 이끕니다.
- `details=summary`는 단계별 선택 요약을 출력하며, `stb`는 표준화 계수를 추가하여 서로 다른 척도의 효과를 비교할 수 있게 합니다.
- `output out=demand_scored p=predicted r=residual`는 적합값과 잔차를 저장하여 이후 스코어링에 사용합니다.

**단계적 선택 요약(Stepwise Selection Summary)**을 탐색 과정의 기록으로 읽어보세요. 전체 12개 효과 모형에서 시작해 한 번에 하나씩 효과를 *제거*하는데, `noise1`, `noise2`, `temp_f`, `Region=Midwest` 대비항, `noise3`를 차례로 제외합니다. 각 제거가 SBC를 낮추기 때문입니다. **모수 추정값(Parameter Estimates)** 표에 남은 것이 최종 선택된 모형입니다.

In [3]:
처리 glmselect 데이터=demand seed=20260531;
    분류 region channel promo / PARAM=reference;
    모형 units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(선택=sbc choose=sbc)
          details=summary stb;
    출력 out=demand_scored p=predicted r=residual;
    라벨 units="주간 판매량" region="지역" channel="판매채널" promo="프로모션"
          price="자사가격" competitor="경쟁사가격" adspend="광고비"
          email="이메일 발송량" temp_f="기온(화씨)" holiday="공휴일 여부"
          noise1="잡음1" noise2="잡음2" noise3="잡음3";
    제목 "수요 동인 변수의 단계적 선택(SBC)";
실행;

                                                    주간 수요 및 후보 동인 변수                                                    

The GLMSELECT Procedure


Dependent Variable: UNITS 주간 판매량


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                Stepwise Selection Summary                                                 

    Step    Action          Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  --------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed             잡음1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136
       2   Removed  


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. 교차검증을 사용한 LASSO

LASSO는 계수를 0 쪽으로 축소시켜 선택과 정규화를 동시에 수행합니다. 고정된 기준에서 멈추는 대신, **5-폴드 교차검증**을 사용하여 LASSO 경로 상에서 표본외 오차가 가장 낮은 지점을 선택하게 합니다.

- `selection=lasso(choose=cv stop=none)`는 전체 LASSO 경로를 추적하고 CV 최적 단계를 선택합니다.
- `cvmethod=random(5)`는 관측치를 5개의 무작위 폴드에 배정합니다.

**LASSO 선택 요약(LASSO Selection Summary)**은 벌점이 완화됨에 따라 효과가 진입하는 순서를 보여줍니다: `price`가 가장 먼저 들어오고, 이어서 `adspend`, `competitor`, 북동부 지역, `email`, `promo`, `holiday` 순입니다—일곱 개의 참 신호가 모두 가짜 변수보다 먼저 진입합니다. 이후 교차검증은 표본외 PRESS가 가장 낮은 단계를 선택하며, 그 결과 **모수 추정값(Parameter Estimates)** 표에는 진짜 동인 변수(및 작은 `Region=Midwest` 항)만 남고, 경로의 맨 마지막에만 진입하는 `temp_f`, `noise1`, `noise2`, `noise3`는 제외됩니다.

In [4]:
처리 glmselect 데이터=demand seed=20260531;
    분류 region channel promo / PARAM=reference;
    모형 units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv 중지=none)
          cvmethod=RANDOM(5);
    라벨 units="주간 판매량" region="지역" channel="판매채널" promo="프로모션"
          price="자사가격" competitor="경쟁사가격" adspend="광고비"
          email="이메일 발송량" temp_f="기온(화씨)" holiday="공휴일 여부"
          noise1="잡음1" noise2="잡음2" noise3="잡음3";
    제목 "5-폴드 교차검증을 활용한 LASSO";
실행;

                                                    주간 수요 및 후보 동인 변수                                                    

The GLMSELECT Procedure


Dependent Variable: UNITS 주간 판매량


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                           LASSO Selection Summary                                                           

    Step    Action               Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  -------------------  -----------------  


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. 홀드아웃으로 검증하는 전진선택법

세 번째 보완 전략: **전진선택법**(효과가 진입만 하고 절대 빠지지 않음)으로 모형을 구축하되, 독립적인 홀드아웃 표본에서 성능이 가장 좋은 지점에서 멈춰 과적합을 직접적으로 방지합니다.

- `partition fraction(validate=0.30)`은 행의 30%를 검증용으로 무작위 예약하여 훈련 70개, 검증 30개 관측치를 남깁니다.
- `selection=forward(select=aic choose=validate)`는 AIC 기준으로 효과를 추가하지만 최종 모형은 검증표본 오차로 선택합니다.

**분할 비율(Partition Fractions)** 표는 70/30 분할을 확인해 줍니다. 이후 전진선택법은 검증 오차가 더 이상 개선되지 않을 때까지 효과를 추가하며, 최종 **모수 추정값(Parameter Estimates)** 표에 있는 여덟 개 효과는 정확히 참 동인 변수이고 네 개의 가짜 변수는 전혀 채택되지 않습니다. 서로 다른 원리에 기반한 세 가지 방법이 동일한 동인 변수에 도달할 때, 그 모형은 단일 선택 규칙의 산물이 아니라 실제일 가능성이 훨씬 높습니다.

In [5]:
처리 glmselect 데이터=demand seed=20260531;
    분류 region channel promo / PARAM=reference;
    모형 units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(선택=aic choose=validate);
    partition FRACTION(validate=0.30);
    라벨 units="주간 판매량" region="지역" channel="판매채널" promo="프로모션"
          price="자사가격" competitor="경쟁사가격" adspend="광고비"
          email="이메일 발송량" temp_f="기온(화씨)" holiday="공휴일 여부"
          noise1="잡음1" noise2="잡음2" noise3="잡음3";
    제목 "30% 홀드아웃으로 검증한 전진선택법";
실행;

                                                    주간 수요 및 후보 동인 변수                                                    

The GLMSELECT Procedure


Dependent Variable: UNITS 주간 판매량


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                          Forward Selection Summary                                                          

    Step    Action               Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  -------------------  --


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. 결과 해석

세 가지 선택 전략 모두 **동일한 참 수요 동인 변수 집합**을 회복하고 모든 가짜 변수를 제외합니다. **모수 추정값(Parameter Estimates)** 표를 직접 읽어보면:

- **가격**이 지배적인 동인 변수입니다. 단계적 선택 모형에서 표준화 계수는 **-0.70**으로 압도적으로 가장 크며, 원래 기울기는 달러당 **-27.8**(단계적 선택 및 LASSO)에서 **-28.6**(전진선택법) 단위 사이에 위치하여, 데이터 생성에 사용된 -28과 거의 정확히 일치합니다. **경쟁사가격**은 세 모형 모두에서 약 **+14.4**로 반대 방향으로 수요를 움직이며, 이는 카테고리 매니저가 기대하는 대체 행동입니다.
- **광고비**(달러당 약 +0.062 단위)와 **이메일 발송량**(발송당 약 +1.2 단위) 모두 판매를 끌어올려 미디어 반응을 정량화합니다.
- **프로모션**과 **공휴일**은 가장 큰 이벤트 효과를 갖습니다: `Promo=No` 대비항은 프로모션이 있는 주 대비 약 **-51에서 -57**이고, 공휴일 상승 효과는 약 **+66에서 +76** 단위입니다. **북동부(Northeast) 지역**(약 +49~+55)과 **온라인(Online) 채널**(약 +24~+32)은 양의 기저 효과를 갖습니다.
- 결정적으로, `temp_f`, `noise1`, `noise2`, `noise3` 등 모든 가짜 변수는 단계적 선택법과 전진선택법에서 **제외**되며 CV로 선택된 LASSO 모형에서도 배제됩니다. 각 방법 모두 구조적 신호를 회복하고 잡음을 무시했습니다.

세 방법이 완전히 동일하지는 않습니다: 단계적 선택법(SBC)과 홀드아웃으로 검증한 전진선택법은 동일한 여덟 개 효과에 정착하는 반면, 교차검증된 LASSO는 작은 `Region=Midwest` 대비항(-8.3, 표준오차 8.3)을 추가로 유지합니다—이는 실질적인 불일치라기보다는 잡음 수준의 차이입니다. 동인 변수 목록이 단계적 SBC, 교차검증 LASSO, 홀드아웃 검증 전진선택법을 모두 통과한다는 것이 진짜 핵심입니다: 서로 다른 세 가지 선택 철학에 견고한 결과는 단일 기준에 맞춰 조정된 결과보다 훨씬 신뢰할 수 있습니다. `OUTPUT OUT=demand_scored`로 적합값과 잔차를 저장해 두면, 동일한 작업 흐름을 다음 분기의 계획된 가격 및 프로모션 일정 스코어링으로 자연스럽게 확장할 수 있습니다.